# Aggregasi dan Pemeriksaan Dataset Twitter/X Kulon Progo

Notebook ini menerima hingga 10 file JSON hasil ekspor Apify yang diletakkan di `data/raw/`. Notebook akan:
1. Membaca dan menggabungkan seluruh file JSON.
2. Memeriksa struktur dan jumlah data dari setiap file.
3. Memeriksa missing values.
4. Memeriksa duplikat berdasarkan `id` (ID post), lalu menghapus duplikat.
5. Menyimpan dataset gabungan yang sudah dibersihkan ke `data/processed/`.
6. Menampilkan ringkasan dan tabel pemeriksaan.

**Catatan:** komentar/replies tetap menjadi bagian dari post induk dan tidak dihitung sebagai post terpisah. Notebook ini tidak mengubah atau mengarang data yang hilang. Pemeriksaan validitas URL secara online tidak dilakukan di notebook ini.

## 1. Import library dan konfigurasi

In [22]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display


RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_JSON = PROCESSED_DIR / "kulonprogo_aggregated.json"
OUTPUT_MISSING = PROCESSED_DIR / "missing_values_report.csv"
OUTPUT_DUPLICATES = PROCESSED_DIR / "duplicates_report.csv"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(RAW_DIR.glob("dataset_*.json"))

print(f"Jumlah dataset ditemukan: {len(json_files)}")

Jumlah dataset ditemukan: 10


## 2. Letakkan 10 file JSON hasil Apify

Simpan file hasil ekspor Apify di `data/raw/`. Contoh nama file:

- `kulonprogo_januari.json`
- `kulonprogo_februari.json`
- ...
- `kulonprogo_file_10.json`

Nama file bebas, asalkan berekstensi `.json`. Notebook membaca semua file JSON yang ditemukan di folder tersebut.

In [23]:
json_files = sorted(RAW_DIR.glob("*.json"))

print(f"Jumlah file JSON ditemukan: {len(json_files)}")
for path in json_files:
    print("-", path.name)

if not json_files:
    raise FileNotFoundError(
        f"Tidak ada file .json di {RAW_DIR.resolve()}. "
        "Masukkan file hasil ekspor Apify ke folder data/raw."
    )

if len(json_files) > 10:
    print("Peringatan: ditemukan lebih dari 10 file. Semua file akan tetap diproses.")

Jumlah file JSON ditemukan: 10
- dataset_1.json
- dataset_10.json
- dataset_2.json
- dataset_3.json
- dataset_4.json
- dataset_5.json
- dataset_6.json
- dataset_7.json
- dataset_8.json
- dataset_9.json


## 3. Membaca dan menggabungkan seluruh file JSON

In [24]:
def extract_records(data):
    """Mengubah format JSON umum menjadi list record post."""
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        # Beberapa ekspor/API membungkus record dalam salah satu key ini.
        for key in ("data", "items", "results", "tweets"):
            if isinstance(data.get(key), list):
                return data[key]
        # Jika objek tunggal tampak seperti satu post, bungkus menjadi list.
        if any(key in data for key in ("id", "tweetId", "text", "fullText")):
            return [data]
    raise ValueError("Struktur JSON tidak dikenali sebagai list post atau objek post.")

all_records = []
file_summary = []

for path in json_files:
    with path.open("r", encoding="utf-8") as f:
        payload = json.load(f)
    records = extract_records(payload)
    all_records.extend(records)
    file_summary.append({
        "nama_file": path.name,
        "jumlah_record": len(records)
    })

file_summary_df = pd.DataFrame(file_summary)
display(file_summary_df)

print("Total record sebelum agregasi:", len(all_records))

,nama_file,jumlah_record
0,dataset_1.json,10
1,dataset_10.json,10
2,dataset_2.json,10
3,dataset_3.json,10
4,dataset_4.json,10
5,dataset_5.json,10
6,dataset_6.json,10
7,dataset_7.json,10
8,dataset_8.json,10
9,dataset_9.json,10


Total record sebelum agregasi: 100


## 4. Membuat DataFrame dan memeriksa struktur

In [25]:
df = pd.json_normalize(all_records, sep=".")
print("Ukuran dataset:", df.shape)
print("Kolom yang ditemukan:")
display(pd.DataFrame({"kolom": df.columns}))

Ukuran dataset: (100, 147)
Kolom yang ditemukan:


,kolom
0,url
1,twitterUrl
2,id
3,text
4,retweetCount
...,...
142,place.url
143,quote.entities.hashtags
144,quote.entities.symbols
145,quote.entities.urls


## 5. Pemeriksaan missing values

Missing value dihitung per kolom. Nilai `None`, `NaN`, dan string kosong dianggap missing. Untuk kolom bertipe list/dict, nilai kosong `[]` atau `{}` tidak dianggap missing karena dapat bermakna bahwa media atau komentar memang tidak tersedia.

In [26]:
def is_missing_value(value):
    if value is None:
        return True
    if isinstance(value, str):
        return value.strip() == ""
    if isinstance(value, (list, dict)):
        return False
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False

missing_rows = []
for col in df.columns:
    missing_count = df[col].map(is_missing_value).sum()
    missing_rows.append({
        "kolom": col,
        "jumlah_missing": int(missing_count),
        "persentase_missing": round(missing_count / len(df) * 100, 2) if len(df) else 0
    })

missing_report = pd.DataFrame(missing_rows).sort_values(
    ["jumlah_missing", "kolom"], ascending=[False, True]
)
display(missing_report)

missing_report.to_csv(
    OUTPUT_MISSING,
    index=False,
    encoding="utf-8-sig"
)

print("Laporan missing values disimpan:", OUTPUT_MISSING)

,kolom,jumlah_missing,persentase_missing
39,author.status,100,100.0
80,quote.author.status,100,100.0
132,quote.author.advertiserAccountType,99,99.0
133,quote.author.analyticsType,99,99.0
135,quote.author.geoEnabled,99,99.0
...,...,...,...
5,replyCount,0,0.0
4,retweetCount,0,0.0
3,text,0,0.0
1,twitterUrl,0,0.0


Laporan missing values disimpan: data\processed\missing_values_report.csv


## 6. Pemeriksaan duplikat berdasarkan ID post

ID post (`id`, atau alternatif `tweetId`) digunakan sebagai kunci deduplikasi. Jika suatu record tidak memiliki ID, record tersebut ditandai agar dapat diperiksa manual dan **tidak dihapus hanya karena ID-nya kosong**.

In [27]:
id_candidates = ["id", "tweetId", "tweet_id"]
id_col = next((col for col in id_candidates if col in df.columns), None)

if id_col is None:
    print("Kolom ID tidak ditemukan. Deduplikasi otomatis tidak dapat dilakukan.")
    df["__dedup_id"] = pd.NA
    duplicate_report = pd.DataFrame(columns=list(df.columns) + ["jumlah_kemunculan"])
    duplicate_mask = pd.Series(False, index=df.index)
else:
    df["__dedup_id"] = df[id_col].map(
        lambda x: str(x).strip() if not is_missing_value(x) else pd.NA
    )
    has_id = df["__dedup_id"].notna()
    duplicate_mask = has_id & df["__dedup_id"].duplicated(keep=False)

    duplicate_report = df.loc[duplicate_mask].copy()
    if not duplicate_report.empty:
        counts = duplicate_report["__dedup_id"].value_counts()
        duplicate_report["jumlah_kemunculan"] = duplicate_report["__dedup_id"].map(counts)

    print("Kolom ID yang digunakan:", id_col)
    print("Jumlah record yang termasuk kelompok duplikat:", int(duplicate_mask.sum()))
    print("Jumlah ID duplikat unik:", int(df.loc[duplicate_mask, "__dedup_id"].nunique()))

display(duplicate_report)

duplicate_report.to_csv(OUTPUT_DUPLICATES, index=False, encoding="utf-8-sig")
print("Laporan duplikat disimpan:", OUTPUT_DUPLICATES)

Kolom ID yang digunakan: id
Jumlah record yang termasuk kelompok duplikat: 60
Jumlah ID duplikat unik: 30


,url,twitterUrl,id,text,retweetCount,replyCount,likeCount,quoteCount,createdAt,bookmarkCount,...,quote.author.professional.category,quote.author.professional.professional_type,quote.author.professional.rest_id,place.url,quote.entities.hashtags,quote.entities.symbols,quote.entities.urls,quote.entities.user_mentions,__dedup_id,jumlah_kemunculan
10,https://x.com/Innosensius_/status/201787256742...,https://twitter.com/Innosensius_/status/201787...,2017872567420522966,Daripada bikin MBG di CFD mending benerin noh ...,0,0,2,0,Sun Feb 01 08:08:04 +0000 2026,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017872567420522966,2
11,https://x.com/ferizandra/status/20176177069258...,https://twitter.com/ferizandra/status/20176177...,2017617706925887908,Ini udah masuk di data keracunan MBG 2025...\n...,15,5,46,0,Sat Jan 31 15:15:21 +0000 2026,13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017617706925887908,2
12,https://x.com/yujultimate_/status/201757139149...,https://twitter.com/yujultimate_/status/201757...,2017571391495909514,@ferizandra Itu sek nang Kulon Progo malah seo...,1,0,12,0,Sat Jan 31 12:11:19 +0000 2026,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017571391495909514,2
13,https://x.com/detikcom/status/2017490781830058503,https://twitter.com/detikcom/status/2017490781...,2017490781830058503,"Kisah Ariendra Ralea Irgantari (9), seorang si...",2,0,8,0,Sat Jan 31 06:51:00 +0000 2026,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017490781830058503,2
14,https://x.com/ferizandra/status/20172967111406...,https://twitter.com/ferizandra/status/20172967...,2017296711140692217,Baru Januari udah ribuan anak yg keracunan...\...,4,1,11,0,Fri Jan 30 17:59:50 +0000 2026,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017296711140692217,2
15,https://x.com/eepphhh/status/2017241043176219027,https://twitter.com/eepphhh/status/20172410431...,2017241043176219027,@infomitigasi Hasil petir tadi jam 4an di Sent...,0,0,1,0,Fri Jan 30 14:18:37 +0000 2026,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017241043176219027,2
16,https://x.com/ferizandra/status/20172206541947...,https://twitter.com/ferizandra/status/20172206...,2017220654194700784,Keracunan MBG di Indonesia 2026 :\n\n08/01/26...,8963,321,18693,501,Fri Jan 30 12:57:36 +0000 2026,991,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017220654194700784,2
17,https://x.com/ResKulonprogo/status/20171221218...,https://twitter.com/ResKulonprogo/status/20171...,2017122121822962130,Kepedulian untuk Masa Depan Generasi Bangsa.\n...,0,4,13,0,Fri Jan 30 06:26:04 +0000 2026,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017122121822962130,2
18,https://x.com/tribunjogja/status/2017029939468...,https://twitter.com/tribunjogja/status/2017029...,2017029939468636266,DLH Kulon Progo merilis hasil Indeks Kualitas ...,0,0,0,0,Fri Jan 30 00:19:46 +0000 2026,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2017029939468636266,2
19,https://x.com/Jateng_Twit/status/2016904546891...,https://twitter.com/Jateng_Twit/status/2016904...,2016904546891813232,Anak Hebat dan Respeck Untuk Gurunya 🥹\nDinas ...,261,34,1109,21,Thu Jan 29 16:01:30 +0000 2026,45,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2016904546891813232,2


Laporan duplikat disimpan: data\processed\duplicates_report.csv


## 7. Menghapus duplikat dan menyimpan dataset agregat

In [28]:
if id_col is not None:
    # Simpan kemunculan pertama untuk setiap ID; record tanpa ID tetap dipertahankan.
    keep_mask = ~(
        df["__dedup_id"].notna()
        & df["__dedup_id"].duplicated(keep="first")
    )
    df_clean = df.loc[keep_mask].copy()
    df_clean = df_clean.drop(columns=["__dedup_id"])
else:
    df_clean = df.copy().drop(columns=["__dedup_id"], errors="ignore")

# Kembalikan ke list of dict agar output berupa JSON array.
clean_records = df_clean.to_dict(orient="records")

with OUTPUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(
        clean_records,
        f,
        ensure_ascii=False,
        indent=4,
        default=str
    )

print("Total record awal :", len(df))
print("Total record akhir:", len(df_clean))
print("Duplikat dihapus  :", len(df) - len(df_clean))
print("File output       :", OUTPUT_JSON.resolve())

Total record awal : 100
Total record akhir: 70
Duplikat dihapus  : 30
File output       : D:\Capstone Ahuy\data\processed\kulonprogo_aggregated.json


## 8. Menampilkan tabel ringkasan post

Tabel berikut dibuat untuk inspeksi. Kolom komentar tetap berada di dalam post induk; jumlah baris tabel merepresentasikan jumlah post, bukan jumlah komentar.

In [29]:
# Ambil nama kolom yang tersedia dari format Apify.
def first_existing(columns):
    return next((c for c in columns if c in df_clean.columns), None)

display_columns = {
    "ID": first_existing(["id", "tweetId", "tweet_id"]),
    "Username": first_existing(["author.userName", "author.username", "username", "userName"]),
    "Post": first_existing(["text", "fullText", "content"]),
    "URL": first_existing(["url", "twitterUrl", "tweetUrl"]),
    "Like": first_existing(["likeCount", "favoriteCount"]),
    "View": first_existing(["viewCount", "views"]),
    "Reply": first_existing(["replyCount", "repliesCount"]),
    "Tanggal": first_existing(["createdAt", "created_at"]),
}

table = pd.DataFrame()
for label, col in display_columns.items():
    if col is not None:
        table[label] = df_clean[col]

if "Post" in table.columns:
    table["Post"] = table["Post"].fillna("").astype(str).str.slice(0, 100)
    table.loc[table["Post"].str.len() == 100, "Post"] += "..."

display(table.head(100))
print(f"Jumlah post unik pada tabel: {len(df_clean)}")

,ID,Username,Post,URL,Like,View,Reply,Tanggal
0,2104251876816703785,NaN,"● Intimate Wedding, salah satu talent Panorama...",https://x.com/panoramaKP/status/21042518768167...,0,NaN,0,Sun Sep 27 16:48:37 +0000 2026
1,2104202181666873379,NaN,7. motoran 4 jam pacitan-kulon progo buat ke a...,https://x.com/plazamayaa/status/21042021816668...,0,NaN,1,Sun Sep 27 13:31:08 +0000 2026
2,2104169595850596750,NaN,UGM dan Pemkab Kulon Progo Berkolaborasi Doron...,https://x.com/berbagikabar20/status/2104169595...,0,NaN,0,Sun Sep 27 11:21:39 +0000 2026
3,2104158871933927820,NaN,Regulasi kalurahan jadi fondasi pengelolaan hu...,https://x.com/RelungIndonesia/status/210415887...,0,NaN,1,Sun Sep 27 10:39:03 +0000 2026
4,2104155970058510836,NaN,🪵 Suka dunia kayu dan furniture? Aquiva – Sent...,https://x.com/lokerjogja_/status/2104155970058...,0,NaN,0,Sun Sep 27 10:27:31 +0000 2026
...,...,...,...,...,...,...,...,...
85,2039202325559210252,BSSN_RI,Badan Siber dan Sandi Negara dan Politeknik Si...,https://x.com/BSSN_RI/status/2039202325559210252,2,284.0,1,Wed Apr 01 04:44:55 +0000 2026
86,2039028207173964010,merapi_uncover,selamat malam min izin saya menyampaikan krono...,https://x.com/merapi_uncover/status/2039028207...,30,11542.0,0,Tue Mar 31 17:13:02 +0000 2026
87,2038925744156418064,mojokdotco,8 Camilan Khas Kulon Progo yang Sebaiknya Jang...,https://x.com/mojokdotco/status/20389257441564...,1,508.0,0,Tue Mar 31 10:25:53 +0000 2026
88,2038906336029934073,indkurniawan,@horni3y Jogja gak punya pantai mbak\nAdanya d...,https://x.com/indkurniawan/status/203890633602...,1,8586.0,2,Tue Mar 31 09:08:46 +0000 2026


Jumlah post unik pada tabel: 70


## 9. Ringkasan akhir

Jalankan sel ini untuk memperoleh ringkasan agregasi, missing values, dan duplikasi.

In [30]:
summary = pd.DataFrame([{
    "jumlah_file_input": len(json_files),
    "jumlah_record_awal": len(df),
    "jumlah_record_duplikat_yang_dihapus": len(df) - len(df_clean),
    "jumlah_post_unik_akhir": len(df_clean),
    "jumlah_post_tanpa_ID": int(
        df[id_col].map(is_missing_value).sum()
    ) if id_col is not None else len(df),
    "jumlah_kolom": len(df_clean.columns),
    "target_100_post_terpenuhi": len(df_clean) >= 100
}])
display(summary)

print("Output:")
print("-", OUTPUT_JSON)
print("-", OUTPUT_MISSING)
print("-", OUTPUT_DUPLICATES)

,jumlah_file_input,jumlah_record_awal,jumlah_record_duplikat_yang_dihapus,jumlah_post_unik_akhir,jumlah_post_tanpa_ID,jumlah_kolom,target_100_post_terpenuhi
0,10,100,30,70,0,147,False


Output:
- data\processed\kulonprogo_aggregated.json
- data\processed\missing_values_report.csv
- data\processed\duplicates_report.csv
